In [2]:
from ttokenizer import Tokenizer
context_length = 12
tokenizer = Tokenizer("tokenizer.json")


### Embedding, sözlük anlamların sayısal değerleri

In [3]:
import torch

torch.manual_seed(1)
embedding = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [4]:
example = "the capital of united states"
example_tokens = tokenizer.encode("the capital of united states")
example_tokens

[0, 61, 1, 61, 2, 61, 3, 61, 4, 58]

In [5]:
example_meaning = embedding(torch.tensor(example_tokens))
example_meaning

tensor([[-1.5256e+00, -7.5023e-01, -6.5398e-01, -1.6095e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.0017e-01, -6.0919e-01, -9.7977e-01, -1.6091e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-7.1214e-01,  3.0372e-01, -7.7731e-01, -2.5145e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-2.2227e-01,  1.6871e+00,  2.2843e-01,  4.6764e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-6.9697e-01, -1.1608e+00,  6.9954e-01,  1.9908e-01],
        [-7.9830e-02,  3.4172e-01,  9.4883e-01, -1.3839e+00]],
       grad_fn=<EmbeddingBackward0>)

In [6]:
example_meaning.shape

torch.Size([10, 4])

In [43]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title, dims=[0, 1, 2]):
    data = [
      go.Scatter3d(
        x=sentence_data["words"][:, dims[0]],
        y=sentence_data["words"][:, dims[1]],
        z=sentence_data["words"][:, dims[2]],
        mode="markers+text",
        marker=dict(
          size=6,
          color=sentence_data["color"],
        ),
        text=sentence_data["labels"],
        hoverinfo="text",
      ) for sentence_data in sentences_data
    ]

    layout = go.Layout(
      scene=dict(
        xaxis_title="Sertlik",
        yaxis_title="Parlaklık",
        zaxis_title="Kırmızılık",
      ),
      title=title,
    )

    fig = go.Figure(data=data, layout=layout)
    plotly.offline.iplot(fig)

In [8]:
tokenizer.tokenizer(example)

['the', ' ', 'capital', ' ', 'of', ' ', 'united', ' ', 'state', 's']

In [9]:
sentences = [
  {
    "words": example_meaning.detach().numpy(),
    "labels": tokenizer.tokenizer(example),
    "color": "red",
  },
]
plot_dots(sentences, "Sözlük V1")

### cümledeki yerine göre embedding

In [29]:

import torch

torch.manual_seed(1)
embeddings = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [26]:
sentence = "the capital of united states and the capital of france"

In [27]:
tokens = tokenizer.encode(sentence)
tokens = torch.tensor(tokens)
tokens.shape


torch.Size([20])

In [30]:
meanings = embeddings(tokens)
meanings

tensor([[-1.5256e+00, -7.5023e-01, -6.5398e-01, -1.6095e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.0017e-01, -6.0919e-01, -9.7977e-01, -1.6091e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-7.1214e-01,  3.0372e-01, -7.7731e-01, -2.5145e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-2.2227e-01,  1.6871e+00,  2.2843e-01,  4.6764e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-6.9697e-01, -1.1608e+00,  6.9954e-01,  1.9908e-01],
        [-7.9830e-02,  3.4172e-01,  9.4883e-01, -1.3839e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.1102e-01,  2.9274e-01, -1.5785e-01, -2.8787e-02],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.5256e+00, -7.5023e-01, -6.5398e-01, -1.6095e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.0017e-01, -6.0919e-01, -9.7977e-01, -1.6091e+00],
        

In [31]:
depend_sentences = [
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "red",
  },
]
plot_dots(depend_sentences, "Depends on Sentence")

### sentence context space

In [16]:
sentence1 = "the dog chased the cat"
sentence2 = "the cat chased the dog"

In [34]:
tokens.shape[0]

20

In [37]:
torch.manual_seed(1)
context_length = 32
position_embeddings = torch.nn.Embedding(num_embeddings=context_length, embedding_dim=4)
positon_meanings = position_embeddings(torch.tensor([i for i in range(tokens.shape[0])]))

In [38]:
positon_meanings

tensor([[-1.5256, -0.7502, -0.6540, -1.6095],
        [-0.1002, -0.6092, -0.9798, -1.6091],
        [-0.7121,  0.3037, -0.7773, -0.2515],
        [-0.2223,  1.6871,  0.2284,  0.4676],
        [-0.6970, -1.1608,  0.6995,  0.1991],
        [ 0.8657,  0.2444, -0.6629,  0.8073],
        [ 1.1017, -0.1759, -2.2456, -1.4465],
        [ 0.0612, -0.6177, -0.7981, -0.1316],
        [ 1.8793, -0.0721,  0.1578, -0.7735],
        [ 0.1991,  0.0457,  0.1530, -0.4757],
        [-0.1110,  0.2927, -0.1578, -0.0288],
        [ 2.3571, -1.0373,  1.5748, -0.6298],
        [-0.9274,  0.5451,  0.0663, -0.4370],
        [ 0.7626,  0.4415,  1.1651,  2.0154],
        [ 0.1374,  0.9386, -0.1860, -0.6446],
        [ 1.5392, -0.8696, -3.3312, -0.7479],
        [-0.0255, -1.0233, -0.5962, -1.0055],
        [-0.2106, -0.0075,  1.6734,  0.0103],
        [-0.7040, -0.1853, -0.9962, -0.8313],
        [-0.4610, -0.5601,  0.3956, -0.9823]], grad_fn=<EmbeddingBackward0>)

In [39]:
meanings_in_context = meanings + positon_meanings
meanings_in_context.shape

torch.Size([20, 4])

In [44]:
depend_sentences = [
  {
    "words": meanings_in_context.detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "red",
  },
]
plot_dots(depend_sentences, "Depends on Sentence")

In [58]:
import math

def get_position_encoding(context_length, embedding_dim, base=10000, device="cpu"):
    p_embeddings = torch.zeros(context_length, embedding_dim, device=device)
    for pos in range(context_length):
        for i in range(0, embedding_dim//2):
            p_embeddings[pos, 2*i] = math.sin(pos/(base**(2*i/embedding_dim)))
            if i+1<embedding_dim:
                p_embeddings[pos, 2*i+1] = math.cos(pos/(base**(2*i+1/embedding_dim)))
    return p_embeddings.unsqueeze(0)

In [61]:
pos_embeddings = get_position_encoding(context_length=20, embedding_dim=4)
pos_embeddings

tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.9950,  0.0100,  1.0000],
         [ 0.9093,  0.9801,  0.0200,  1.0000],
         [ 0.1411,  0.9553,  0.0300,  1.0000],
         [-0.7568,  0.9211,  0.0400,  1.0000],
         [-0.9589,  0.8776,  0.0500,  1.0000],
         [-0.2794,  0.8253,  0.0600,  1.0000],
         [ 0.6570,  0.7648,  0.0699,  1.0000],
         [ 0.9894,  0.6967,  0.0799,  1.0000],
         [ 0.4121,  0.6216,  0.0899,  1.0000],
         [-0.5440,  0.5403,  0.0998,  1.0000],
         [-1.0000,  0.4536,  0.1098,  1.0000],
         [-0.5366,  0.3624,  0.1197,  1.0000],
         [ 0.4202,  0.2675,  0.1296,  1.0000],
         [ 0.9906,  0.1700,  0.1395,  1.0000],
         [ 0.6503,  0.0707,  0.1494,  1.0000],
         [-0.2879, -0.0292,  0.1593,  1.0000],
         [-0.9614, -0.1288,  0.1692,  1.0000],
         [-0.7510, -0.2272,  0.1790,  1.0000],
         [ 0.1499, -0.3233,  0.1889,  1.0000]]])

In [62]:
pos_embeddings.shape

torch.Size([1, 20, 4])

In [64]:
meanings_in_sentence = meanings + pos_embeddings
meanings_in_sentence

tensor([[[-1.5256,  0.2498, -0.6540, -0.6095],
         [ 0.0966,  0.7928, -0.2197,  1.0013],
         [ 0.8091,  0.3709, -0.9598, -0.6091],
         [-0.6037,  0.7532, -0.1997,  1.0013],
         [-1.4689,  1.2248, -0.7373,  0.7485],
         [-1.7038,  0.6754, -0.1797,  1.0013],
         [-0.5017,  2.5124,  0.2884,  1.4676],
         [-0.0879,  0.5627, -0.1597,  1.0013],
         [ 0.2924, -0.4641,  0.7795,  1.1991],
         [ 0.3323,  0.9633,  1.0387, -0.3839],
         [-1.2889,  0.3381, -0.1298,  1.0013],
         [-1.1110,  0.7463, -0.0481,  0.9712],
         [-1.2814,  0.1602, -0.1100,  1.0013],
         [-1.1054, -0.4827, -0.5243, -0.6095],
         [ 0.2458, -0.0322, -0.0901,  1.0013],
         [ 0.5501, -0.5385, -0.8303, -0.6091],
         [-1.0327, -0.2314, -0.0703,  1.0013],
         [-1.6735,  0.1749, -0.6081,  0.7485],
         [-1.4958, -0.4294, -0.0506,  1.0013],
         [ 2.0292, -0.3954,  0.3466,  0.2265]]], grad_fn=<AddBackward0>)

In [66]:
sentences_with_pos = [
  {
    "words": meanings_in_sentence[0].detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "red",
  },
]
plot_dots(sentences_with_pos, "Sentence Context Space V2")

In [67]:
sentences = [
  {
    "words": meanings_in_sentence[0].detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "red",
  },
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "blue",
  },
]
plot_dots(sentences, "Sentence Context Space V2")

### Rotary Position Embedding - Rope

In [ ]:
freqs_indices = torch.arange(0, 20, dtype=torch.float32)

freqs = 1.0/(10000 ** (freqs_indices/20))
freqs.unsqueeze(1) # birinci index'e boyut ekler. 20 satır, 1 kolon
freqs.unsqueeze(0) # sıfırıncı index'e yeni boyut ekler. 1 satır, 20 kolon
# cumlenin sonuna dogru frekans azaliyor

(tensor([[1.0000e+00],
         [6.3096e-01],
         [3.9811e-01],
         [2.5119e-01],
         [1.5849e-01],
         [1.0000e-01],
         [6.3096e-02],
         [3.9811e-02],
         [2.5119e-02],
         [1.5849e-02],
         [1.0000e-02],
         [6.3096e-03],
         [3.9811e-03],
         [2.5119e-03],
         [1.5849e-03],
         [1.0000e-03],
         [6.3096e-04],
         [3.9811e-04],
         [2.5119e-04],
         [1.5849e-04]]),
 tensor([[1.0000e+00, 6.3096e-01, 3.9811e-01, 2.5119e-01, 1.5849e-01, 1.0000e-01,
          6.3096e-02, 3.9811e-02, 2.5119e-02, 1.5849e-02, 1.0000e-02, 6.3096e-03,
          3.9811e-03, 2.5119e-03, 1.5849e-03, 1.0000e-03, 6.3096e-04, 3.9811e-04,
          2.5119e-04, 1.5849e-04]]))

In [91]:
# input -> context_length + embedding_dim
# frekanslarla pozisyonlari carparak acilari bulma
def get_rotary_position_encoding(input: torch.Tensor, base=10000, device="cpu"):
    context_length, dimension = input.shape #32, 4
    assert dimension % 2 == 0
    half_dimension = dimension // 2
    freqs_indices = torch.arange(0, half_dimension, device=device, dtype=torch.float32)
    
    freqs = 1.0/(base ** (freqs_indices/dimension))
    
    positions = torch.arange(0, context_length, device=device, dtype=torch.float32).unsqueeze(1)
    
    angles = positions * freqs
    
    sin_angles = torch.sin(angles)
    cos_angles = torch.cos(angles)
    
    input_even = input[:, 0:dimension:2] # [0,2,4,..]
    input_odd  = input[:, 1:dimension:2] # [1,3,5,7,..]

    input_even_rotated = input_even * cos_angles - input_odd * sin_angles
    input_odd_rotated = input_even * sin_angles + input_odd * cos_angles 
    
    input_rotated = torch.empty_like(input)
    input_rotated[:, 0:dimension:2] = input_even_rotated
    input_rotated[:, 1:dimension:2] = input_odd_rotated
    
    return input_rotated

    
    
    
    
    

In [90]:
torch.manual_seed(1)
random_input = torch.randn(context_length,20)
pos_rotary_encodings = get_rotary_position_encoding(random_input)


In [92]:
meanings.shape

torch.Size([20, 4])

In [93]:
meanings_with_pos_encodings = get_rotary_position_encoding(meanings)

In [94]:
meanings_with_pos_encodings

tensor([[-1.5256, -0.7502, -0.6540, -1.6095],
        [-0.2323, -0.7360, -0.2287, -0.0216],
        [ 0.5956,  0.1624, -0.6406, -1.7717],
        [ 0.7659,  0.0950, -0.2198, -0.0666],
        [ 0.6953,  0.3404, -0.6180, -0.5343],
        [-0.4051,  0.6569, -0.2022, -0.1089],
        [ 0.2580,  1.6820, -0.0755,  0.5149],
        [-0.4287, -0.6418, -0.1765, -0.1469],
        [ 1.2498, -0.5207,  0.3446,  0.6405],
        [-0.0681, -0.3443,  1.6739, -0.1170],
        [ 0.5150,  0.5748, -0.1252, -0.1925],
        [ 0.2922,  0.1123, -0.0459, -0.1537],
        [-0.7370,  0.2291, -0.0845, -0.2136],
        [-1.0692, -1.3218,  1.3759, -1.0607],
        [ 0.0984, -0.7655, -0.0403, -0.2261],
        [ 0.4722,  0.3977,  1.5358, -1.0911],
        [ 0.6551,  0.4080,  0.0054, -0.2296],
        [ 0.4880,  0.6011,  0.3495, -0.7384],
        [-0.6436,  0.4259,  0.0509, -0.2240],
        [ 1.8689,  0.2104,  0.6809,  0.3993]], grad_fn=<CopySlices>)

In [97]:
sentences_with_rotary = [
  {
    "words": meanings_with_pos_encodings.detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "red",
  },
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenizer(sentence),
    "color": "blue"
  }
]
plot_dots(sentences_with_rotary, "Sentence Context Space V3 Rotary Emcoding")

In [98]:
meanings_with_pos_encodings, meanings

(tensor([[-1.5256, -0.7502, -0.6540, -1.6095],
         [-0.2323, -0.7360, -0.2287, -0.0216],
         [ 0.5956,  0.1624, -0.6406, -1.7717],
         [ 0.7659,  0.0950, -0.2198, -0.0666],
         [ 0.6953,  0.3404, -0.6180, -0.5343],
         [-0.4051,  0.6569, -0.2022, -0.1089],
         [ 0.2580,  1.6820, -0.0755,  0.5149],
         [-0.4287, -0.6418, -0.1765, -0.1469],
         [ 1.2498, -0.5207,  0.3446,  0.6405],
         [-0.0681, -0.3443,  1.6739, -0.1170],
         [ 0.5150,  0.5748, -0.1252, -0.1925],
         [ 0.2922,  0.1123, -0.0459, -0.1537],
         [-0.7370,  0.2291, -0.0845, -0.2136],
         [-1.0692, -1.3218,  1.3759, -1.0607],
         [ 0.0984, -0.7655, -0.0403, -0.2261],
         [ 0.4722,  0.3977,  1.5358, -1.0911],
         [ 0.6551,  0.4080,  0.0054, -0.2296],
         [ 0.4880,  0.6011,  0.3495, -0.7384],
         [-0.6436,  0.4259,  0.0509, -0.2240],
         [ 1.8689,  0.2104,  0.6809,  0.3993]], grad_fn=<CopySlices>),
 tensor([[-1.5256e+00, -7.5023e-01, 

## Modelling


In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united states and the capital of france"
tokens = tokenizer.encode(prompt)
model = MasterModel(vocab_size=len(tokenizer.vocab), embedding_dim=4, context_length=32)


In [2]:
model

MasterModel(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
)

In [6]:
model(tokens)

tensor([[-0.4238, -0.0254, -0.2017, -0.8997],
        [ 1.4043,  1.7783,  1.4974, -0.0246],
        [ 1.6574, -0.6210,  0.5099,  1.1832],
        [-2.2014,  0.5369,  1.4725,  0.2734],
        [ 0.4113,  0.3310, -0.6604,  0.7146],
        [ 0.4279, -2.2252,  1.3888,  0.5605],
        [ 1.2507,  0.8541,  0.0394,  0.3198],
        [ 1.8453,  1.3150,  1.2498,  0.8252],
        [-1.6255,  0.4993, -0.5685, -0.8293],
        [ 1.4156,  1.1106, -0.7871,  1.3494],
        [-2.0124, -1.0415,  0.9501,  1.1577],
        [ 1.0366, -1.5281, -0.1734,  0.6853],
        [ 1.7845, -1.3965,  0.7011,  1.3234],
        [-0.3739, -0.2011,  0.8129, -0.4351],
        [ 0.5272,  2.2037,  0.4242,  1.4363],
        [ 1.7650,  0.1328, -1.0037,  0.8078],
        [-2.2232, -0.4377,  0.1304,  1.4919],
        [ 0.2341,  0.4732, -0.8652, -0.4452],
        [ 1.3232, -1.8394, -0.1686,  1.4881],
        [-0.8395, -0.6032, -0.5442, -0.0142]], grad_fn=<CopySlices>)